<a href="https://colab.research.google.com/github/leandroclv/polars_udemy/blob/main/condicional_manipulacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import polars as pl

In [2]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df = pl.read_csv(url)
df.head()

total_bill,tip,sex,smoker,day,time,size
f64,f64,str,str,str,str,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4


In [3]:
df2 = (df.with_columns(
    pl.when(pl.col('tip') > 5)
      .then(pl.lit('alta'))
      .otherwise(pl.lit('normal'))
      .alias('categoria_gorjeta'))
    .sort('tip', descending=True)
)
df2.select(['total_bill', 'tip', 'categoria_gorjeta']).head()

total_bill,tip,categoria_gorjeta
f64,f64,str
50.81,10.0,"""alta"""
48.33,9.0,"""alta"""
39.42,7.58,"""alta"""
48.27,6.73,"""alta"""
34.3,6.7,"""alta"""


In [4]:
df3 = df.with_columns([
    pl.col('sex').str.to_lowercase().alias('sexo_lower'),
    pl.col('day').str.to_uppercase().alias('dia_upper'),
    pl.col('day').str.replace('Sun', 'Domingo').alias('dia_replace'),
    pl.col('day').str.contains('u').alias('contem_u')
])

df3.select(['sex', 'sexo_lower', 'dia_upper', 'dia_replace', 'contem_u']).head()

sex,sexo_lower,dia_upper,dia_replace,contem_u
str,str,str,str,bool
"""Female""","""female""","""SUN""","""Domingo""",true
"""Male""","""male""","""SUN""","""Domingo""",true
"""Male""","""male""","""SUN""","""Domingo""",true
"""Male""","""male""","""SUN""","""Domingo""",true
"""Female""","""female""","""SUN""","""Domingo""",true


In [4]:
df = pl.DataFrame({
    'codigo': ['ID123', 'REF4528', 'X789']
})

df.with_columns(pl.col('codigo').str.extract(r'(\d+)', group_index=1).alias('numeros'))

codigo,numeros
str,str
"""ID123""","""123"""
"""REF4528""","""4528"""
"""X789""","""789"""


In [6]:
df = pl.DataFrame({'nome': ['ana', 'carlos', 'miguel', 'joão']})
df.slice(1,2)

nome
str
"""carlos"""
"""miguel"""


In [8]:
df = pl.DataFrame({'texto': ['foo_bar', 'abc_def_ghi']})
df.with_columns(pl.col('texto').str.split('_').alias('coluna_split'))

texto,coluna_split
str,list[str]
"""foo_bar""","[""foo"", ""bar""]"
"""abc_def_ghi""","[""abc"", ""def"", ""ghi""]"


In [9]:
import datetime as dt

In [10]:
df4 = pl.DataFrame({'evento': ['início', 'meio', 'fim'],
                    'data': [dt.datetime(2026,1,1),dt.datetime(2026,6,1), dt.datetime(2026,12,31)]
                    })

In [11]:
df4

evento,data
str,datetime[μs]
"""início""",2026-01-01 00:00:00
"""meio""",2026-06-01 00:00:00
"""fim""",2026-12-31 00:00:00


In [13]:
df4 = df4.with_columns([
    pl.col('data').dt.year().alias('ano'),
    pl.col('data').dt.month().alias('mes'),
    pl.col('data').dt.weekday().alias('dia_semana'),
    (pl.col('data')+ pl.duration(days=30)).alias('data_mais_30_dias')
])

df4

evento,data,ano,mes,dia_semana,data_mais_30_dias
str,datetime[μs],i32,i8,i8,datetime[μs]
"""início""",2026-01-01 00:00:00,2026,1,4,2026-01-31 00:00:00
"""meio""",2026-06-01 00:00:00,2026,6,1,2026-07-01 00:00:00
"""fim""",2026-12-31 00:00:00,2026,12,4,2027-01-30 00:00:00


In [14]:
#list e struct
df5 = pl.DataFrame({
    'nome': ['Ana', 'Patricia', 'Rodrigo'],
    'notas': [[9.5, 8.7, 9.0], [6.5,9.7,7.0], [8.5,7.7,6.0]]
})

In [15]:
#Acessar elementos dentro da lista
df5 = df5.with_columns(
    pl.col('notas').list.mean().alias('qtd_notas')
)

df5

nome,notas,qtd_notas
str,list[f64],f64
"""Ana""","[9.5, 8.7, 9.0]",9.066667
"""Patricia""","[6.5, 9.7, 7.0]",7.733333
"""Rodrigo""","[8.5, 7.7, 6.0]",7.4


In [17]:
#Struct

df6 = pl.DataFrame({
    'id': [1,2],
    'info': [
     {'cidade': 'São Paulo', 'idade': 32},
     {'cidade': 'Curitiba', 'idade': 29}]
})

In [18]:
df6

id,info
i64,struct[2]
1,"{""São Paulo"",32}"
2,"{""Curitiba"",29}"


In [19]:
df6 = df6.with_columns(
    pl.col('info').struct.field('cidade').alias('cidade')
)

df6

id,info,cidade
i64,struct[2],str
1,"{""São Paulo"",32}","""São Paulo"""
2,"{""Curitiba"",29}","""Curitiba"""
